In [ ]:
import torch
import pandas as pd
from IPython.display import display
from tslearn.datasets import UCR_UEA_datasets
from sklearn.preprocessing import LabelEncoder
from uni2ts.model.moirai import MoiraiModule
from encoder import MoiraiEncoder

from heads import *
from utils import *

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PATCH_SIZES = [8, 16, 32, 64]
SIZE = "small"
NUM_VARS = 6
DO_MASK = True
KEEP_MASK_EMBEDDING = True

/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ds = UCR_UEA_datasets()
X_train, y_train, X_test, y_test = ds.load_dataset("LSST")

le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

num_classes = len(set(y_train))

X_train_target_tensor, X_train_is_target_observed, X_train_is_target_padded = preprocess_data(X_train, device=DEVICE)
X_test_target_tensor, X_test_is_target_observed, X_test_is_target_padded = preprocess_data(X_test, device=DEVICE)

y_train_tensor = torch.tensor(y_train, dtype=torch.long, device=DEVICE)
y_test_tensor = torch.tensor(y_test, dtype=torch.long, device=DEVICE)

In [ ]:
Z_train_patch = {}
Z_test_patch = {}
for patch in PATCH_SIZES:
    
    MODEL_PARAMS = {
        "patch_size": patch,
        "num_samples": 100,
        "target_dim": 6,
        "feat_dynamic_real_dim": 0,
        "past_feat_dynamic_real_dim": 0,
    }
    if DO_MASK == True:
        prediction_length = patch
    else:
        prediction_length = 0
    encoder = MoiraiEncoder(
        module=MoiraiModule.from_pretrained(f"Salesforce/moirai-1.1-R-{SIZE}"),
        prediction_length=prediction_length,
        context_length=36,
        **MODEL_PARAMS,
    )
    encoder.eval()
    encoder.to(DEVICE)
    
    with torch.no_grad():
        Z_train = encoder(
            past_target=X_train_target_tensor,
            past_observed_target=X_train_is_target_observed,
            past_is_pad=X_train_is_target_padded,
        )
        Z_test = encoder(
            past_target=X_test_target_tensor,
            past_observed_target=X_test_is_target_observed,
            past_is_pad=X_test_is_target_padded,
        )
        
    if DO_MASK and not KEEP_MASK_EMBEDDING:
        N, S, F = Z_train.shape
        P = S // NUM_VARS
        
        Z_train_reshaped = Z_train.view(N, NUM_VARS, P, F)
        Z_test_reshaped = Z_test.view(Z_test.shape[0], NUM_VARS, P, F)
        
        Z_train_no_mask = Z_train_reshaped[:, :, :-1, :]
        Z_test_no_mask = Z_test_reshaped[:, :, :-1, :]
        
        Z_train = Z_train_no_mask.reshape(N, -1, F)
        Z_test = Z_test_no_mask.reshape(Z_test.shape[0], -1, F)



    Z_train_patch[patch] = Z_train
    Z_test_patch[patch] = Z_test

# Reproduce mean pooling with pytorch

In [ ]:
df_nn_linear = pd.DataFrame(index=["Mean Pooling NN"], columns=PATCH_SIZES + ['Multipatch'])
df_nn_linear.columns.name = "Patch Size"

for p in PATCH_SIZES:
    tr_loader, va_loader, te_loader = create_single_scale_dataloaders(
        Z_train_patch[p], Z_test_patch[p], y_train_tensor, y_test_tensor, batch_size=256, device=DEVICE
    )
    _, acc = grid_search_attention_model(
        model_class=MeanPoolingClassifier,
        model_kwargs={"num_vars": NUM_VARS, "num_classes": num_classes},
        train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE
    )
    df_nn_linear.loc["Mean Pooling NN", p] = acc

SCALES = [64, 32, 16, 8]
patches_counts = {s: Z_train_patch[s].shape[1] // NUM_VARS for s in SCALES}
tr_loader, va_loader, te_loader = create_all_scales_dataloaders(
    Z_train_patch, Z_test_patch, SCALES, y_train_tensor, y_test_tensor, batch_size=256, device=DEVICE
)

_, acc_multi = grid_search_attention_model(
    model_class=MultiScaleMeanPoolingClassifier,
    model_kwargs={"num_vars": NUM_VARS, "num_classes": num_classes, "patches_counts": patches_counts},
    train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE
)

df_nn_linear.loc["Mean Pooling NN", "Multipatch"] = acc_multi

In [ ]:
df_nn_linear.astype(float).round(4)

Patch Size,8,16,32,64,Multipatch
Mean Pooling NN,0.6095,0.6204,0.6054,0.635,0.6614


# First attention

In [6]:
PATCH_SIZES_TO_TEST = [8, 16]
MODES = ["shared_context", "independent_context"]
NUM_HEADS = 16

model_names_single = [
    "SingleScale Attention (Base)", 
    f"SingleScale MHA (Heads={NUM_HEADS})"
]

index_single = pd.MultiIndex.from_product([model_names_single, MODES], names=["Modèle", "Mode"])
df_results_single = pd.DataFrame(index=index_single, columns=PATCH_SIZES_TO_TEST)
df_results_single.columns.name = "Patch Size"

for patch in PATCH_SIZES_TO_TEST:
    tr_loader, va_loader, te_loader = create_single_scale_dataloaders(
        Z_train_patch[patch], Z_test_patch[patch], y_train_tensor, y_test_tensor, batch_size=256, device=DEVICE
    )
    
    for mode in MODES:
        print(f"\n--- PATCH: {patch} | MODE: {mode} ---")
        
        # 1. Attention Simple (Base)
        _, test_acc = grid_search_attention_model(
            model_class=SingleScaleAttentionClassifier,
            model_kwargs={"num_vars": NUM_VARS, "num_classes": num_classes, "mode": mode},
            train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE
        )
        df_results_single.loc[(model_names_single[0], mode), patch] = test_acc

        # 2. Multi-Head Attention
        _, test_acc = grid_search_attention_model(
            model_class=SingleScaleMultiHeadClassifier,
            model_kwargs={"num_vars": NUM_VARS, "num_classes": num_classes, "patch_mode": mode, "num_heads": NUM_HEADS},
            train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE
        )
        df_results_single.loc[(model_names_single[1], mode), patch] = test_acc


--- PATCH: 8 | MODE: shared_context ---

--- PATCH: 8 | MODE: independent_context ---

--- PATCH: 16 | MODE: shared_context ---

--- PATCH: 16 | MODE: independent_context ---


In [7]:
display(df_results_single.astype(float).round(4))

Patch Size                                            8       16
Modèle                       Mode                               
SingleScale Attention (Base) shared_context       0.5912  0.6062
                             independent_context  0.6083  0.6038
SingleScale MHA (Heads=16)   shared_context       0.6164  0.6172
                             independent_context  0.6111  0.6245

# Nb of heads

In [ ]:
PATCH = 16 
MODES = ["shared_context", "independent_context"]
HEADS_TO_TEST = [4, 8, 16, 32, 64, 128, 384] 

df_heads_results = pd.DataFrame(index=MODES, columns=HEADS_TO_TEST)
df_heads_results.index.name = "Mode"
df_heads_results.columns.name = f"Num Heads (Patch {PATCH})"

tr_loader, va_loader, te_loader = create_single_scale_dataloaders(
    Z_train_patch[PATCH], Z_test_patch[PATCH], y_train_tensor, y_test_tensor, batch_size=256, device=DEVICE
)


for mode in MODES:
    for heads in HEADS_TO_TEST:
        print(f"\n MODE: {mode} | HEADS: {heads}")
        
        _, test_acc = grid_search_attention_model(
            model_class=SingleScaleMultiHeadClassifier,
            model_kwargs={"num_vars": NUM_VARS, "num_classes": num_classes, "patch_mode": mode, "num_heads": heads},
            train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE
        )
        
        df_heads_results.loc[mode, heads] = test_acc


 MODE: shared_context | HEADS: 16

 MODE: shared_context | HEADS: 32

 MODE: shared_context | HEADS: 64

 MODE: shared_context | HEADS: 128

 MODE: shared_context | HEADS: 384

 MODE: independent_context | HEADS: 16

 MODE: independent_context | HEADS: 32

 MODE: independent_context | HEADS: 64

 MODE: independent_context | HEADS: 128

 MODE: independent_context | HEADS: 384


In [9]:
display(df_heads_results.astype(float).round(4))

Num Heads (Patch 16),16,32,64,128,384
Mode,,,,,
shared_context,0.6180,0.6233,0.6245,0.6217,0.6229
independent_context,0.6302,0.6212,0.6172,0.6249,0.6212


# Hierarchical

In [ ]:
PATCH_SIZES_TO_TEST = [8, 16]
MODES = ["shared_context", "independent_context"]
NUM_HEADS = 4

df_results_hierarchical = pd.DataFrame(index=MODES, columns=PATCH_SIZES_TO_TEST)
df_results_hierarchical.index.name = "Mode"
df_results_hierarchical.columns.name = "Patch Size"

for patch in PATCH_SIZES_TO_TEST:
    tr_loader, va_loader, te_loader = create_single_scale_dataloaders(
        Z_train_patch[patch], Z_test_patch[patch], y_train_tensor, y_test_tensor, batch_size=256, device=DEVICE
    )
    
    for mode in MODES:
        print(f"\nPATCH: {patch} | MODE: {mode}")
        
        _, test_acc = grid_search_attention_model(
            model_class=HierarchicalMultiHeadClassifier,
            model_kwargs={"num_vars": NUM_VARS, "num_classes": num_classes, "patch_mode": mode, "num_heads": NUM_HEADS},
            train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE
        )
        df_results_hierarchical.loc[mode, patch] = test_acc


--- PATCH: 8 | MODE: shared_context ---

--- PATCH: 8 | MODE: independent_context ---

--- PATCH: 16 | MODE: shared_context ---

--- PATCH: 16 | MODE: independent_context ---


In [11]:
display(df_results_hierarchical.astype(float).round(4))

Patch Size,8,16
Mode,,
shared_context,0.5819,0.5868
independent_context,0.6079,0.5864


# Multiscale

In [ ]:
SCALES = [64, 32, 16, 8]
patches_counts = {s: Z_train_patch[s].shape[1] // NUM_VARS for s in SCALES}

tr_loader, va_loader, te_loader = create_all_scales_dataloaders(
    Z_train_patch, Z_test_patch, SCALES, y_train_tensor, y_test_tensor, batch_size=256, device=DEVICE
)

experiences = [
    ("Séquentiel (Cascade)", SequentialCrossScaleClassifier, True),   # 1 Couche Partagée
    ("Séquentiel (Cascade)", SequentialCrossScaleClassifier, False),  # 3 Couches Perso
    ("Parallèle (Concat)", ParallelCrossScaleClassifier, True),       # 1 Couche Partagée
    ("Parallèle (Concat)", ParallelCrossScaleClassifier, False)       # 3 Couches Perso
]

results_multi = pd.DataFrame(index=["Séquentiel (Cascade)", "Parallèle (Concat)"], 
                             columns=["shared", "indepedent"])

for arch_name, model_class, shared in experiences:
    col_name = "shared" if shared else "indepedent"
    
    _, test_acc = grid_search_attention_model(
        model_class=model_class,
        model_kwargs={
            "num_vars": NUM_VARS,
            "num_classes": num_classes,
            "patches_counts": patches_counts,
            "num_heads": 64,
            "shared_layer": shared
        },
        train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE
    )
    
    results_multi.loc[arch_name, col_name] = test_acc


🔄 Test en cours : Séquentiel (Cascade) | 1 Couche Partagée

🔄 Test en cours : Séquentiel (Cascade) | 3 Couches Perso

🔄 Test en cours : Parallèle (Concat) | 1 Couche Partagée

🔄 Test en cours : Parallèle (Concat) | 3 Couches Perso


In [13]:
display(results_multi.astype(float).round(4))

,1 Couche Partagée,3 Couches Perso
Séquentiel (Cascade),0.6245,0.6476
Parallèle (Concat),0.6346,0.6436
